### Import Libraries

In [ ]:
import pandas as pd
import numpy as np


### Load Data

In [ ]:
df = pd.read_csv("movies_dataset.csv")

### Display first 5 rows

In [ ]:
df.head(5)

In [ ]:
df.columns

### Clean Data

In [ ]:
df = df[["title", "overview", "poster_path", "vote_average", "release_date"]]
df.isna().sum()

In [ ]:
df["vote_average"] = df["vote_average"].fillna(
    df["vote_average"].mean()
)

df["overview"] = df["overview"].fillna("")
df["poster_path"] = df["poster_path"].fillna("")
df["release_date"] = df["release_date"].fillna("")

In [ ]:
df["vote_average"] = df["vote_average"].fillna(df["vote_average"].mean())

In [ ]:
df.isna().sum()

In [ ]:
# df.fillna("",inplace=True)

In [ ]:
df.isna().sum()

In [ ]:
df = df.drop_duplicates()

In [ ]:
df['combined'] = (
    df['title'] + " " +
    df['overview']
)

In [ ]:
df['combined']

### Convert text into numbers

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
vectorizer = TfidfVectorizer(stop_words='english')
matrix = vectorizer.fit_transform(df['combined'])

In [ ]:
similarity  =  cosine_similarity(matrix)
similarity

In [ ]:
def recommend(movie_name):

    movie_name = movie_name.lower().strip()

    if movie_name not in df["title"].str.lower().values:
        return "Movie not found"

    idx = df[df["title"].str.lower() == movie_name].index[0]

    scores = list(enumerate(similarity[idx]))

    scores = sorted(
        scores,
        key=lambda x: x[1],
        reverse=True
    )[1:6]

    recommendations = []

    for i in scores:

        movie = df.iloc[i[0]]

        recommendations.append({
            "title": movie["title"],
            "rating": movie["vote_average"],
            "poster_path": movie["poster_path"]
        })

    return recommendations

In [ ]:
user_input = input("Enter movie name").title()


In [ ]:
user_input

In [ ]:
user_input = user_input.replace("'",'')
user_input

In [ ]:
recommend(user_input)

In [ ]:
import pickle
# Reduce memory usage
similarity = similarity.astype("float32")
with open("movie_data.pkl", "wb") as file:
    pickle.dump((df, similarity), file)

In [ ]:
print(df.shape)
print(similarity.shape)
print(type(similarity))

In [ ]:
print(similarity.dtype)
print(similarity.nbytes / (1024 ** 2), "MB")